# Estimate mineable land

**Terminology stack** (kept consistent through downstream notebooks):

| Layer | Definition | Output |
|---|---|---|
| 1. **Available land** | Land that passes categorical exclusions (not water / urban / existing mine / aeolian sand / protected area / Antarctica). | `available-land-1km.tif` |
| 2. **Mineable land** | Available land that *additionally* meets `mining_suitability ≥ τ`, where τ is the **area-weighted P50 of mining-suitability observed at OSM aggregate quarries** — an empirical anchor calibrated to where quarrying actually happens. | `mineable-land-1km.tif` |
| 3. **Aggregate reserve** | (Notebook 4) Mineable land × lithology factor `T·ρ·r`. L/M/H scenarios live here, driven by lithology parameters only. | `aggregate-reserve-1km-{L,M,H}.tif` |

In CRIRSCO/JORC terms, the chain corresponds to an **accessible / mineable reserve**.

## Why a single threshold (no L/M/H here)

The earlier design varied τ across L/M/H (P75/P50/P25 of OSM). When combined with the lithology L/M/H in notebook 4, this produced a **double-coupled** sensitivity that blew up the L–H spread to ~30× (BAU trade went from 0.5 Gt/yr at M to 9 Gt/yr at L — implausible). 

**Decoupling fixes this**: the threshold is the *empirical anchor* (a single calibrated value), and L/M/H reflects the *engineering uncertainty in extractability parameters* (T, r) — which is what the L/M/H label is meant to capture in mining literature anyway.

Expected post-decoupling spread: L:M:H ≈ 1 : 2.3 : 3.7 of reserve, driven purely by lithology factor (T_M × 0.5..1.5, r_M ± 0.05–0.10).

## What this notebook does

1. Build available land (categorical exclusions).
2. Compute the area-weighted percentile of mining-suitability at OSM quarries (used for diagnostic and to pin τ at P50).
3. (Optional) Sweep arbitrary τ to inspect the area / reserve trade-off.
4. Per-country diagnostic at τ = P50.
5. Save a **single** mineable-land raster at τ = P50.

In [1]:
import numpy as np
import pandas as pd
import rasterio as rio
from rasterio.warp import reproject, Resampling
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
(RESULTS / 'mining-area').mkdir(parents=True, exist_ok=True)

## Step 1 — Available land (categorical exclusions)

Exclude: urban, existing quarries, water, aeolian sand deserts (GSDS30 classes 1–2), all IUCN protected areas (Ia–VI), cells outside any country, and Antarctica (Antarctic Treaty).

In [3]:
country_raster_1km_  = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')
country_raster_50km_ = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-50km.tif')
country_1km          = country_raster_1km_.read(1)

urban_1km   = rio.open(RESULTS / 'features' / 'urban-area-2020-1km.tif').read(1)
mining_1km  = rio.open(RESULTS / 'quarry-area-rasters' / 'global-quarry-area-raster-1km.tif').read(1)
water_1km   = rio.open(RESULTS / 'features' / 'water-mask-1km.tif').read(1)

desert_frac_1km = (
    rio.open(RESULTS / 'GSDS30-1km' / 'class1-frac-1km.tif').read(1)
    + rio.open(RESULTS / 'GSDS30-1km' / 'class2-frac-1km.tif').read(1)
)
desert_frac_1km = np.clip(desert_frac_1km, 0, 1).astype(np.float32)

PA_mask = np.clip(
    rio.open(RESULTS / 'protected-area-rasters' / 'protected-area-Ia-mask.tif').read(1)
  + rio.open(RESULTS / 'protected-area-rasters' / 'protected-area-Ib-mask.tif').read(1)
  + rio.open(RESULTS / 'protected-area-rasters' / 'protected-area-II-mask.tif').read(1)
  + rio.open(RESULTS / 'protected-area-rasters' / 'protected-area-III-mask.tif').read(1)
  + rio.open(RESULTS / 'protected-area-rasters' / 'protected-area-IV-mask.tif').read(1)
  + rio.open(RESULTS / 'protected-area-rasters' / 'protected-area-V-mask.tif').read(1)
  + rio.open(RESULTS / 'protected-area-rasters' / 'protected-area-VI-mask.tif').read(1),
    0, 1,
)

In [4]:
available_land_1km = (
    (1 - urban_1km / 100)
    * (1 - mining_1km)
    * (1 - water_1km)
    * (1 - desert_frac_1km)
).astype(np.float32)

available_land_1km[PA_mask == 1]      = 0
available_land_1km[country_1km == 0]  = 0
available_land_1km[country_1km == 213] = 0   # Antarctica (Antarctic Treaty)

print(f'Available land: {available_land_1km.sum() / 1e6:.2f} M km^2')

Available land: 109.46 M km^2


In [5]:
def _save(path, arr, template):
    with rio.open(
        path, 'w', driver='GTiff',
        height=arr.shape[0], width=arr.shape[1],
        count=1, dtype=arr.dtype,
        crs=template.crs, transform=template.transform,
        compress='lzw',
    ) as dst:
        dst.write(arr, 1)

def _upscale_to_50km(arr_1km):
    arr_50km = np.zeros(country_raster_50km_.shape, dtype=np.float32)
    reproject(
        source=arr_1km, destination=arr_50km,
        src_transform=country_raster_1km_.transform, src_crs=country_raster_1km_.crs,
        dst_transform=country_raster_50km_.transform, dst_crs=country_raster_50km_.crs,
        resampling=Resampling.sum,
    )
    return np.clip(arr_50km, 0, 2500)

_save(RESULTS / 'mining-area' / 'available-land-1km.tif',  available_land_1km, country_raster_1km_)
_save(RESULTS / 'mining-area' / 'available-land-50km.tif', _upscale_to_50km(available_land_1km), country_raster_50km_)

## Step 2 — Calibrate τ against OSM aggregate quarries

**Distribution of suitability where OSM aggregate quarries actually exist**, both cell-counted and area-weighted (large quarries count more). Reveals what counts as a "realistic" suitability level for aggregate operations.

In [6]:
suit_1km   = rio.open(RESULTS / 'global-mining-suitability-1km.tif').read(1)
quarry_1km = rio.open(RESULTS / 'quarry-area-rasters' / 'global-aggregate-quarry-area-raster-1km.tif').read(1)

qmask = quarry_1km > 0
print(f'OSM aggregate quarry cells: {qmask.sum():,}  |  total area {quarry_1km[qmask].sum():.1f} (raster units)')

OSM aggregate quarry cells: 197,341  |  total area 19005.5 (raster units)


In [7]:
suit_at_q = suit_1km[qmask].astype(np.float64)
w         = quarry_1km[qmask].astype(np.float64)
order     = np.argsort(suit_at_q)
cum_w     = np.cumsum(w[order]) / w.sum() * 100

perc = [5, 10, 25, 50, 75, 90, 95]
pct_table = pd.DataFrame({
    'percentile' : [f'P{p}' for p in perc],
    'cell-counted'  : [np.percentile(suit_at_q, p) for p in perc],
    'area-weighted' : [suit_at_q[order][np.searchsorted(cum_w, p)] for p in perc],
}).set_index('percentile')
pct_table.loc['mean'] = [suit_at_q.mean(), np.nan]
pct_table.style.format('{:.0f}')

,cell-counted,area-weighted
percentile,,
P5,2372,2611
P10,3954,4195
P25,6350,6337
P50,8219,8240
P75,9305,9298
P90,9685,9659
P95,9812,9783
mean,7456,nan


In [8]:
# What fraction of observed OSM quarries survives at each candidate threshold?
rows = []
total_cells = qmask.sum()
total_area  = quarry_1km[qmask].sum()
for tau in range(4000, 9501, 500):
    keep = qmask & (suit_1km >= tau)
    rows.append({
        'tau'              : tau,
        'OSM_cells_kept_%' : 100 * keep.sum() / total_cells,
        'OSM_area_kept_%'  : 100 * quarry_1km[keep].sum() / total_area,
    })
osm_coverage = pd.DataFrame(rows)
osm_coverage.style.format({'OSM_cells_kept_%': '{:.1f}', 'OSM_area_kept_%': '{:.1f}'})

,tau,OSM_cells_kept_%,OSM_area_kept_%
0,4000,89.8,90.9
1,4500,87.7,88.6
2,5000,85.1,85.4
3,5500,81.8,81.9
4,6000,78.0,78.0
5,6500,73.6,73.5
6,7000,68.2,68.3
7,7500,61.8,61.7
8,8000,53.9,54.1
9,8500,44.7,45.2


## Step 3 — Sweep τ: mineable area & reserve preview

For each candidate τ, report:
- Remaining mineable land (M km², % of available)
- Global aggregate reserve at M (central) scenario, in Tt (preview only — full L/M/H is produced by notebook 4)

Use this together with the OSM coverage table above to pick τ.

In [9]:
# Reproject GLiM to 1-km Eckert IV and build M-scenario factor lookup (Mt per km^2).
GLiM_ = rio.open(DATA / 'GLiM.tif')
GLiM_src = GLiM_.read(1)
GLiM_1km = np.zeros(country_raster_1km_.shape, dtype=np.uint8)
reproject(
    source=GLiM_src, destination=GLiM_1km,
    src_transform=GLiM_.transform, src_crs=GLiM_.crs,
    dst_transform=country_raster_1km_.transform, dst_crs=country_raster_1km_.crs,
    resampling=Resampling.nearest,
)
del GLiM_src

# code, T (m), rho (t/m^3), r  -- M scenario; ev/wb/ig set to 0
params_M = [
    ( 1, 10, 1.80, 0.70), ( 2, 40, 2.90, 0.75), ( 3, 30, 2.40, 0.65), ( 4, 50, 2.95, 0.80),
    ( 5, 30, 2.50, 0.65), ( 6, 40, 2.60, 0.75), ( 7, 40, 2.55, 0.75), ( 8, 40, 2.75, 0.75),
    ( 9, 50, 2.70, 0.80), (10, 40, 2.70, 0.75), (11,  0, 0.00, 0.00), (12, 20, 2.20, 0.60),
    (13, 50, 2.80, 0.80), (14,  0, 0.00, 0.00), (15, 30, 2.50, 0.70), (16,  0, 0.00, 0.00),
]
factor_M = np.zeros(17, dtype=np.float32)
for code, T, rho, r in params_M:
    factor_M[code] = T * rho * r
factor_per_cell_M = factor_M[GLiM_1km]

In [10]:
thresholds = list(range(5000, 9501, 500))
base_Mkm2  = available_land_1km.sum() / 1e6
base_Tt    = (available_land_1km * factor_per_cell_M).sum() / 1e6

rows = [{'tau': 'no filter', 'mineable_Mkm2': base_Mkm2, 'pct_of_available': 100.0, 'reserve_M_Tt': base_Tt}]
for t in thresholds:
    mineable = available_land_1km * (suit_1km >= t)
    rows.append({
        'tau'               : t,
        'mineable_Mkm2'     : mineable.sum() / 1e6,
        'pct_of_available'  : 100 * mineable.sum() / available_land_1km.sum(),
        'reserve_M_Tt'      : (mineable * factor_per_cell_M).sum() / 1e6,
    })
sweep = pd.DataFrame(rows)
sweep.style.format({
    'mineable_Mkm2'    : '{:,.2f}',
    'pct_of_available' : '{:.1f}',
    'reserve_M_Tt'     : '{:,.0f}',
})

,tau,mineable_Mkm2,pct_of_available,reserve_M_Tt
0,no filter,109.46,100.0,"5,685"
1,5000,20.05,18.3,"1,129"
2,5500,16.50,15.1,931
3,6000,13.38,12.2,759
4,6500,10.67,9.7,607
5,7000,8.36,7.6,478
6,7500,6.34,5.8,365
7,8000,4.58,4.2,264
8,8500,3.07,2.8,178
9,9000,1.83,1.7,110


## Step 4 — Per-country diagnostic at M (central) threshold

Check how mineable land contracts in a few representative countries when using `τ = P50` (the central M scenario). Useful for sanity-checking that resource-rich countries don't shrink too aggressively and that small / dense countries still keep some land.

In [11]:
# Use the M scenario's threshold (P50, area-weighted) for the per-country diagnostic.
tau_M = int(pct_table.loc['P50', 'area-weighted'])
print(f'M scenario uses τ = {tau_M}  (= area-weighted P50 of OSM)')

pseudo   = pd.read_csv(DATA / 'country-pseudo-values.csv')
iso2code = dict(zip(pseudo['country'], pseudo['value']))

mineable_M = available_land_1km * (suit_1km >= tau_M)

watch = ['CHN','USA','SAU','AUS','RUS','IND','BRA','CAN','DEU','JPN','EGY','NZL','SGP']
diag = []
for iso in watch:
    if iso not in iso2code:
        continue
    m   = country_1km == iso2code[iso]
    base = available_land_1km[m].sum() / 1e3
    mine = mineable_M[m].sum() / 1e3
    diag.append({
        'iso'           : iso,
        'available_kkm2': base,
        'mineable_kkm2' : mine,
        'shrink_%'      : 100 * (1 - mine / base) if base > 0 else float('nan'),
    })
pd.DataFrame(diag).sort_values('shrink_%', ascending=False).reset_index(drop=True)

M scenario uses τ = 8240  (= area-weighted P50 of OSM)


,iso,available_kkm2,mineable_kkm2,shrink_%
0,SAU,988.466248,3.878905,99.607582
1,AUS,4512.051758,33.094116,99.266541
2,CHN,8580.585938,82.906303,99.033791
3,CAN,7660.944824,102.021690,98.668289
4,BRA,6924.645020,108.891075,98.427490
5,EGY,560.968445,8.952896,98.404022
6,IND,2887.984131,56.047070,98.059303
7,JPN,273.369934,5.862646,97.855415
8,RUS,14665.034180,572.231201,96.097984
9,NZL,189.186371,7.566675,96.000412


## Step 5 — Save mineable land

Single τ = **area-weighted P50** of OSM aggregate-quarry suitability. No L/M/H here — that axis lives in notebook 4 (lithology only).

In [12]:
# Single threshold: area-weighted P50 of OSM aggregate quarries.
tau = int(pct_table.loc['P50', 'area-weighted'])
print(f'Mineable-land threshold: τ = {tau}  (= area-weighted P50 of OSM)')

mineable_1km  = (available_land_1km * (suit_1km >= tau)).astype(np.float32)
mineable_50km = _upscale_to_50km(mineable_1km)

out_1km  = RESULTS / 'mining-area' / 'mineable-land-1km.tif'
out_50km = RESULTS / 'mining-area' / 'mineable-land-50km.tif'
_save(out_1km,  mineable_1km,  country_raster_1km_)
_save(out_50km, mineable_50km, country_raster_50km_)

print(f'  Mineable land: {mineable_1km.sum() / 1e6:.2f} M km^2')
print(f'  Saved: {out_1km.name}, {out_50km.name}')

Mineable-land threshold: τ = 8240  (= area-weighted P50 of OSM)
  Mineable land: 3.82 M km^2
  Saved: mineable-land-1km.tif, mineable-land-50km.tif
